# Dice Loss vs. Dice+Focal Loss: Class Imbalance Handling for Tumor Segmentation

This notebook demonstrates how using a combined Dice+Focal loss function (as proposed in the report) can better address class imbalance compared to standard Dice loss.

Two pipelines are trained and compared on toy/patchwise BRATS segmentation data:
- **Regular Dice loss**
- **Combined Dice + Focal loss**

Results (validation Dice, loss curves) highlight robustness on minority classes.



In [20]:
# Setup, imports, and config
import os, glob, random
import numpy as np
import SimpleITK as sitk
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt

DATA_ROOT = "/kaggle/input/brats2015/BRATS2015/training"
RANDOM_SEED = 42
PATCH_SIZE = (64,64,64)
N_CASES = 12
PATCHES_PER_CASE = 10
BATCH_SIZE = 2
EPOCHS = 8
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [21]:
# Utility: normalization, loader for all 4 modalities, z-score per channel
def zscore_norm(img):
    mask = img > 0
    if mask.sum() == 0:
        return (img - img.mean()) / (img.std() + 1e-8)
    else:
        mean = img[mask].mean()
        std = img[mask].std()
        return (img - mean) / (std + 1e-8)

def load_mha_case(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.mha")))
    t1 = next(f for f in files if "t1." in f.lower() and not "t1c" in f.lower())
    t1ce = next(f for f in files if "t1c" in f.lower())
    t2 = next(f for f in files if "t2." in f.lower())
    flair = next(f for f in files if "flair" in f.lower())
    seg = next(f for f in files if "ot." in f.lower())
    vols = []
    for f in [t1, t2, t1ce, flair]:
        img = sitk.GetArrayFromImage(sitk.ReadImage(f)).astype(np.float32)
        img = zscore_norm(img)
        vols.append(img)
    vol = np.stack(vols, axis=0)  # [4, D, H, W]
    seg = sitk.GetArrayFromImage(sitk.ReadImage(seg)).astype(np.int16)
    return vol, seg

# Dataset with tumor-centric sampling (reduces empty-class bias)
class BRATSPatchDataset(Dataset):
    def __init__(self, data_root, n_cases=N_CASES, patch=PATCH_SIZE, patches_per_case=PATCHES_PER_CASE):
        folders = sorted(glob.glob(os.path.join(data_root, "*/*/")))
        random.shuffle(folders)
        selected = folders[:n_cases]
        self.patch = patch
        self.patches_per_case = patches_per_case
        self.volumes, self.segs = [], []
        print(f"Loading {len(selected)} cases...")
        for case in selected:
            try:
                v, l = load_mha_case(case)
                self.volumes.append(v)
                self.segs.append(l)
            except Exception as e:
                print(f"Skipping {case} due to {e}")
        print("Done loading.")
    def __len__(self):
        return len(self.volumes) * self.patches_per_case
    def __getitem__(self, idx):
        cidx = idx // self.patches_per_case
        img4 = self.volumes[cidx]
        seg = self.segs[cidx]
        D,H,W = img4.shape[1:]
        pd,ph,pw=self.patch
        tumor_mask = (seg > 0)
        if tumor_mask.any() and random.random() < 0.7:
            tz, ty, tx = np.argwhere(tumor_mask)[np.random.randint(0, tumor_mask.sum())]
            z0 = max(0, min(int(tz) - pd//2, D-pd))
            y0 = max(0, min(int(ty) - ph//2, H-ph))
            x0 = max(0, min(int(tx) - pw//2, W-pw))
        else:
            z0 = random.randint(0, max(1, D-pd))
            y0 = random.randint(0, max(1, H-ph))
            x0 = random.randint(0, max(1, W-pw))
        patch4 = img4[:,z0:z0+pd,y0:y0+ph,x0:x0+pw]    # [4, pd, ph, pw]
        fused = patch4.mean(axis=0, keepdims=True)      # [1, pd, ph, pw]
        label = seg[z0:z0+pd,y0:y0+ph,x0:x0+pw]
        fused = torch.from_numpy(fused).float()
        label = torch.from_numpy(label).long()
        return fused, label



In [22]:
# --- Simple ResUNet3D ---
class ResidualBlock3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        identity = self.proj(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, in_ch, 3, stride=2, padding=1)
        self.rb = ResidualBlock3d(in_ch, out_ch)
    def forward(self, x):
        x = self.down(x)
        return self.rb(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.rb = ResidualBlock3d(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        ds = [skip.size(d) - x.size(d) for d in range(2,5)]
        if any(ds):
            x = nn.functional.pad(x, [0, ds[2], 0, ds[1], 0, ds[0]])
        x = torch.cat((x, skip), 1)
        return self.rb(x)

class SimpleResUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=5, base=8):
        super().__init__()
        self.stem = ResidualBlock3d(in_ch, base)
        self.down1 = DownBlock(base, base*2)
        self.down2 = DownBlock(base*2, base*4)
        self.down3 = DownBlock(base*4, base*8)
        self.down4 = DownBlock(base*8, base*8)
        self.up1 = UpBlock(base*8, base*4, base*8)
        self.up2 = UpBlock(base*4, base*2, base*4)
        self.up3 = UpBlock(base*2, base, base*2)
        self.up4 = UpBlock(base, base, base)
        self.head = nn.Conv3d(base, out_ch, 1)
    def forward(self, x):
        s1 = self.stem(x)
        d1 = self.down1(s1)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        u1 = self.up1(d4, d3)
        u2 = self.up2(u1, d2)
        u3 = self.up3(u2, d1)
        u4 = self.up4(u3, s1)
        return self.head(u4)



In [23]:
# --- Loss Functions ---
def dice_loss(pred, target, num_classes=5):
    pred_soft = torch.softmax(pred, dim=1)
    onehot = torch.nn.functional.one_hot(target, num_classes=num_classes).permute(0,4,1,2,3).float()
    intersect = (pred_soft * onehot).sum(dim=(2,3,4))
    denom = pred_soft.sum(dim=(2,3,4)) + onehot.sum(dim=(2,3,4))
    loss = 1 - (2*intersect+1) / (denom+1)
    return loss.mean()

def focal_loss(pred, target, num_classes=5, gamma=2.0, alpha=None):
    # Stable per-voxel focal loss with optional class weighting (alpha)
    logpt = torch.nn.functional.log_softmax(pred, dim=1)  # [B,C,D,H,W]
    pt = logpt.exp()
    onehot = torch.nn.functional.one_hot(target, num_classes=num_classes).permute(0,4,1,2,3).float()
    # Per-voxel alpha weights from class vector
    if alpha is None:
        alpha_vec = torch.ones(num_classes, device=pred.device, dtype=pred.dtype)
    else:
        alpha_vec = alpha.to(pred.device).float().view(-1)
    # Broadcast alpha to voxels via onehot → at: [B,D,H,W]
    at = (alpha_vec.view(1, num_classes, 1, 1, 1) * onehot).sum(dim=1)
    # Select pt/logpt for true classes: [B,D,H,W]
    logpt_t = (logpt * onehot).sum(dim=1)
    pt_t = (pt * onehot).sum(dim=1)
    loss = -at * ((1.0 - pt_t) ** gamma) * logpt_t
    return loss.mean()

def combined_dice_focal_loss(pred, target, gamma=2.0, alpha=None):
    d = dice_loss(pred, target)
    f = focal_loss(pred, target, gamma=gamma, alpha=alpha)
    return d + f

In [24]:
# Evaluation dataset (deterministic center crop) and per-class Dice over full loader
class BRATSEvalPatchDataset(Dataset):
    def __init__(self, data_root, n_cases=4, patch=PATCH_SIZE):
        folders = sorted(glob.glob(os.path.join(data_root, "*/*/")))
        selected = folders[-n_cases:]
        self.volumes, self.segs = [], []
        self.patch = patch
        for case in selected:
            try:
                v, l = load_mha_case(case)
                self.volumes.append(v)
                self.segs.append(l)
            except Exception as e:
                print(f"Skipping eval case {case}: {e}")
    def __len__(self):
        return len(self.volumes)
    def __getitem__(self, idx):
        img4 = self.volumes[idx]
        seg = self.segs[idx]
        D,H,W = img4.shape[1:]
        pd,ph,pw = self.patch
        z0 = max(0, (D - pd)//2); y0 = max(0, (H - ph)//2); x0 = max(0, (W - pw)//2)
        patch4 = img4[:, z0:z0+pd, y0:y0+ph, x0:x0+pw]
        fused = patch4.mean(axis=0, keepdims=True)
        label = seg[z0:z0+pd, y0:y0+ph, x0:x0+pw]
        return torch.from_numpy(fused).float(), torch.from_numpy(label).long()

def eval_per_class_dice(model, loader, num_classes=5):
    model.eval()
    inter = torch.zeros(num_classes, dtype=torch.float64, device=DEVICE)
    denom = torch.zeros(num_classes, dtype=torch.float64, device=DEVICE)
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = torch.argmax(model(xb), dim=1)
            for c in range(num_classes):
                p = (pred == c); t = (yb == c)
                inter[c] += (p & t).sum().to(dtype=torch.float64)
                denom[c] += (p.sum() + t.sum()).to(dtype=torch.float64)
    dice = torch.where(denom > 0, (2*inter+1e-4)/(denom+1e-4), torch.tensor(float('nan'), device=DEVICE))
    return dice.cpu().tolist()

def macro_tumor_dice(per_class_dice_list):
    # classes 1..4 are tumor subregions
    vals = [v for v in per_class_dice_list[1:5] if (v==v)]  # drop NaNs
    return float(np.mean(vals)) if len(vals)>0 else float('nan')


In [25]:
# Build loaders
train_loader = DataLoader(BRATSPatchDataset(DATA_ROOT), batch_size=BATCH_SIZE, shuffle=True)
eval_loader  = DataLoader(BRATSEvalPatchDataset(DATA_ROOT),  batch_size=1, shuffle=False)

Loading 12 cases...
Done loading.


In [26]:
# --- Training Loop: Compare Dice vs. Dice+Focal Loss (full-eval per-class each epoch) ---
model1 = SimpleResUNet3D(in_ch=1, out_ch=5, base=8).to(DEVICE)  # Dice
model2 = SimpleResUNet3D(in_ch=1, out_ch=5, base=8).to(DEVICE)  # Dice+Focal
opt1 = torch.optim.AdamW(model1.parameters(), lr=1e-3)
opt2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)
class_names = ["Background", "Necrosis", "Edema", "Non-enh Tumor", "Enh Tumor"]
alpha = torch.tensor([0.1, 2.0, 2.0, 3.0, 3.0], dtype=torch.float32, device=DEVICE)

print('Training Standard Dice Loss Pipeline:')
for epoch in range(EPOCHS):
    model1.train(); total_loss = []
    for x, y in tqdm(train_loader, desc=f"DiceLoss Epoch {epoch+1}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model1(x)
        loss = dice_loss(out, y)
        opt1.zero_grad(); loss.backward(); opt1.step()
        total_loss.append(loss.item())
    d1 = eval_per_class_dice(model1, eval_loader, num_classes=5)
    print(f"DiceLoss Epoch {epoch+1}: loss={np.mean(total_loss):.4f}")
    for cn, v in zip(class_names, d1):
        print(f"{cn:>12}: {v:.3f}")

print('\nTraining Combined Dice+Focal Loss Pipeline:')
for epoch in range(EPOCHS):
    model2.train(); total_loss = []
    for x, y in tqdm(train_loader, desc=f"DiceFocal Epoch {epoch+1}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model2(x)
        loss = combined_dice_focal_loss(out, y, gamma=2.0, alpha=alpha)
        opt2.zero_grad(); loss.backward(); opt2.step()
        total_loss.append(loss.item())
    d2 = eval_per_class_dice(model2, eval_loader, num_classes=5)
    print(f"Dice+Focal Epoch {epoch+1}: loss={np.mean(total_loss):.4f}")
    for cn, v in zip(class_names, d2):
        print(f"{cn:>12}: {v:.3f}")
    print("— Comparison this epoch —")
    for cn, a, b in zip(class_names, d1, d2):
        print(f"{cn:>12}: Dice={a:.3f} | Dice+Focal={b:.3f}")

Training Standard Dice Loss Pipeline:


DiceLoss Epoch 1: 100%|██████████| 60/60 [00:10<00:00,  5.48it/s]


DiceLoss Epoch 1: loss=0.8556
  Background: 0.745
    Necrosis: 0.000
       Edema: 0.221
Non-enh Tumor: 0.000
   Enh Tumor: 0.049


DiceLoss Epoch 2: 100%|██████████| 60/60 [00:10<00:00,  5.69it/s]


DiceLoss Epoch 2: loss=0.7161
  Background: 0.891
    Necrosis: 0.000
       Edema: 0.186
Non-enh Tumor: 0.000
   Enh Tumor: 0.000


...

Dice+Focal Epoch 6: loss=0.9577
  Background: 0.908
    Necrosis: 0.000
       Edema: 0.394
Non-enh Tumor: 0.000
   Enh Tumor: 0.198
— Comparison this epoch —
  Background: Dice=0.956 | Dice+Focal=0.908
    Necrosis: Dice=0.000 | Dice+Focal=0.000
       Edema: Dice=0.000 | Dice+Focal=0.394
Non-enh Tumor: Dice=0.000 | Dice+Focal=0.000
   Enh Tumor: Dice=0.000 | Dice+Focal=0.198


DiceFocal Epoch 7: 100%|██████████| 60/60 [00:11<00:00,  5.41it/s]


Dice+Focal Epoch 7: loss=0.9770
  Background: 0.697
    Necrosis: 0.000
       Edema: 0.203
Non-enh Tumor: 0.000
   Enh Tumor: 0.163
— Comparison this epoch —
  Background: Dice=0.956 | Dice+Focal=0.697
    Necrosis: Dice=0.000 | Dice+Focal=0.000
       Edema: Dice=0.000 | Dice+Focal=0.203
Non-enh Tumor: Dice=0.000 | Dice+Focal=0.000
   Enh Tumor: Dice=0.000 | Dice+Focal=0.163


DiceFocal Epoch 8: 100%|██████████| 60/60 [00:10<00:00,  5.46it/s]

Dice+Focal Epoch 8: loss=0.9469
  Background: 0.266
    Necrosis: 0.000
       Edema: 0.133
Non-enh Tumor: 0.000
   Enh Tumor: 0.176
— Comparison this epoch —
  Background: Dice=0.956 | Dice+Focal=0.266
    Necrosis: Dice=0.000 | Dice+Focal=0.000
       Edema: Dice=0.000 | Dice+Focal=0.133
Non-enh Tumor: Dice=0.000 | Dice+Focal=0.000
   Enh Tumor: Dice=0.000 | Dice+Focal=0.176
